In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from ultralytics import YOLO
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# ── Paths ──
PROJECT_ROOT = '/mnt/Data/AKIB/YOLO-OD-IM'
DATA_YAML    = os.path.join(PROJECT_ROOT, 'dataset', 'data_abs.yaml')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']
nc = cfg['nc']
print(f"Classes: {nc}, Names: {class_names[:5]}...")

## 1.1 Train Baseline YOLOv11n (nano) – Quick Baseline

In [ ]:
# ── Train a quick baseline with YOLOv11n at 640x640 (default settings) ──
model_n = YOLO('yolo11n.pt')

results_n = model_n.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,                    # maps to physical GPU 1 via CUDA_VISIBLE_DEVICES
    project=os.path.join(PROJECT_ROOT, 'runs'),
    name='baseline_yolo11n',
    exist_ok=True,
    patience=20,                 # early stopping
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

## 1.2 Evaluate Baseline on Test Set

In [ ]:
# ── Evaluate baseline model on test set ──
best_path = os.path.join(PROJECT_ROOT, 'runs', 'baseline_yolo11n', 'weights', 'best.pt')
model_baseline = YOLO(best_path)

# Default conf=0.25, iou=0.7 — typical YOLO defaults
metrics_baseline = model_baseline.val(
    data=DATA_YAML,
    split='test',
    device=0,
    conf=0.25,
    iou=0.7,
    plots=True,
    project=os.path.join(PROJECT_ROOT, 'runs'),
    name='baseline_eval',
    exist_ok=True,
)

print("\n" + "="*60)
print("BASELINE RESULTS (YOLOv11n, conf=0.25, iou=0.7)")
print("="*60)
print(f"  mAP@50:      {metrics_baseline.box.map50:.4f}")
print(f"  mAP@50-95:   {metrics_baseline.box.map:.4f}")
print(f"  Precision:   {metrics_baseline.box.mp:.4f}")
print(f"  Recall:      {metrics_baseline.box.mr:.4f}")
print("="*60)

In [ ]:
# ── Per-class Recall Analysis ──
per_class_recall = metrics_baseline.box.r   # shape: (nc,)
per_class_precision = metrics_baseline.box.p

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Recall per class
sorted_idx_r = np.argsort(per_class_recall)
axes[0].barh(range(len(per_class_recall)), per_class_recall[sorted_idx_r], color='steelblue')
axes[0].set_yticks(range(len(per_class_recall)))
axes[0].set_yticklabels([class_names[i] for i in sorted_idx_r], fontsize=5)
axes[0].set_xlabel('Recall')
axes[0].set_title('Per-Class Recall (Baseline YOLOv11n)')
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='50% threshold')
axes[0].legend()

# Precision per class
sorted_idx_p = np.argsort(per_class_precision)
axes[1].barh(range(len(per_class_precision)), per_class_precision[sorted_idx_p], color='coral')
axes[1].set_yticks(range(len(per_class_precision)))
axes[1].set_yticklabels([class_names[i] for i in sorted_idx_p], fontsize=5)
axes[1].set_xlabel('Precision')
axes[1].set_title('Per-Class Precision (Baseline YOLOv11n)')
axes[1].axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='50% threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'baseline_per_class_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

# Identify zero-recall classes
zero_recall = [class_names[i] for i in range(nc) if per_class_recall[i] == 0]
low_recall = [class_names[i] for i in range(nc) if 0 < per_class_recall[i] < 0.3]
print(f"\nClasses with ZERO recall: {len(zero_recall)} → {zero_recall}")
print(f"Classes with recall < 30%: {len(low_recall)} → {low_recall}")

## 1.3 Quick Win: Confidence Threshold Sweep

The biggest quick win — lowering confidence threshold from 0.25 to 0.1-0.15 can boost recall by **+5-15%** immediately.

In [ ]:
# ── Confidence Threshold Sweep ──
conf_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
sweep_results = []

for conf in conf_thresholds:
    m = model_baseline.val(
        data=DATA_YAML,
        split='test',
        device=0,
        conf=conf,
        iou=0.7,
        verbose=False,
    )
    sweep_results.append({
        'conf': conf,
        'precision': m.box.mp,
        'recall': m.box.mr,
        'map50': m.box.map50,
        'map': m.box.map,
    })
    print(f"conf={conf:.2f} → P={m.box.mp:.4f}, R={m.box.mr:.4f}, mAP50={m.box.map50:.4f}")

# Plot Precision-Recall trade-off
fig, ax = plt.subplots(figsize=(10, 6))
confs = [r['conf'] for r in sweep_results]
precs = [r['precision'] for r in sweep_results]
recs  = [r['recall'] for r in sweep_results]

ax.plot(confs, precs, 'o-', color='coral', label='Precision', linewidth=2)
ax.plot(confs, recs, 's-', color='steelblue', label='Recall', linewidth=2)
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('Precision vs Recall at Different Confidence Thresholds (Baseline)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.savefig(os.path.join(PROJECT_ROOT, 'conf_threshold_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

## 1.4 NMS IoU Threshold Sweep

For densely packed shelf products, increasing IoU threshold from 0.7 to 0.5-0.6 can help detect overlapping items.

In [ ]:
# ── NMS IoU Threshold Sweep ──
best_conf = 0.15  # from the sweep above, pick the best conf
iou_thresholds = [0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8]
iou_results = []

for iou_t in iou_thresholds:
    m = model_baseline.val(
        data=DATA_YAML,
        split='test',
        device=0,
        conf=best_conf,
        iou=iou_t,
        verbose=False,
    )
    iou_results.append({
        'iou': iou_t,
        'precision': m.box.mp,
        'recall': m.box.mr,
        'map50': m.box.map50,
    })
    print(f"iou={iou_t:.2f} → P={m.box.mp:.4f}, R={m.box.mr:.4f}, mAP50={m.box.map50:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ious = [r['iou'] for r in iou_results]
precs = [r['precision'] for r in iou_results]
recs  = [r['recall'] for r in iou_results]

ax.plot(ious, precs, 'o-', color='coral', label='Precision', linewidth=2)
ax.plot(ious, recs, 's-', color='steelblue', label='Recall', linewidth=2)
ax.set_xlabel('NMS IoU Threshold')
ax.set_ylabel('Metric Value')
ax.set_title(f'P/R vs NMS IoU Threshold (conf={best_conf})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.savefig(os.path.join(PROJECT_ROOT, 'iou_threshold_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary of Baseline Findings ──
print("\n" + "="*70)
print("📋 BASELINE EVALUATION SUMMARY")
print("="*70)
print(f"\nModel: YOLOv11n (nano) — smallest model")
print(f"Dataset: {nc} classes, 924 train / 40 val / 35 test images")
print(f"\nDefault settings (conf=0.25, iou=0.7):")
print(f"  Precision: {sweep_results[4]['precision']:.4f}")
print(f"  Recall:    {sweep_results[4]['recall']:.4f}")
print(f"  mAP@50:    {sweep_results[4]['map50']:.4f}")
print(f"\nBest recall at conf=0.15:")
print(f"  Precision: {sweep_results[2]['precision']:.4f}")
print(f"  Recall:    {sweep_results[2]['recall']:.4f}")
print(f"  mAP@50:    {sweep_results[2]['map50']:.4f}")
print(f"\n→ Next steps: Train with YOLOv11m/l, class balancing, 1280 resolution")